# 📚 Unidad 2: Material Complementario - Teoría
## Módulo 02 - Mapas y Visualización Geoespacial
### Laboratorio - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Dominar visualización geoespacial con Plotly y H3
2. ✅ Crear mapas interactivos (scatter, choropleth, heatmaps)
3. ✅ Aplicar indexación jerárquica con H3
4. ✅ Análisis espacial para decisiones de negocio

---

## 1️⃣ Introducción a Mapas con Plotly

### Tipos de Mapas

**1. Scatter Mapbox**: Puntos en mapa
**2. Density Mapbox**: Heatmap de densidad
**3. Choropleth**: Áreas coloreadas (polígonos)
**4. Line Mapbox**: Rutas y trayectorias

### Setup

```python
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import h3

# Token Mapbox (opcional, para estilos premium)
# px.set_mapbox_access_token('tu_token_aqui')
```

---

## 2️⃣ Scatter Mapbox (Puntos)

```python
fig = px.scatter_mapbox(
    df,
    lat='latitud',
    lon='longitud',
    color='categoria',
    size='ventas',
    hover_name='nombre',
    hover_data=['direccion', 'ventas'],
    title='Sucursales por Ventas',
    mapbox_style='open-street-map',  # Gratis
    zoom=11,
    center={'lat': -32.89, 'lon': -68.85}  # Mendoza
)
fig.show()
```

**Estilos disponibles (gratis):**
* `open-street-map`
* `carto-positron`
* `carto-darkmatter`
* `stamen-terrain`

---

## 3️⃣ H3: Indexación Geoespacial

### ¿Qué es H3?

**H3** es un sistema de indexación geoespacial jerárquico desarrollado por Uber que divide el mundo en hexágonos.

**Ventajas:**
* ✅ Agregación rápida por zona
* ✅ Jerarquía (zoom in/out)
* ✅ Vecindad fácil de calcular
* ✅ Área uniforme por resolución

### Resoluciones H3

| Resolución | Área aprox | Uso |
|-----------|-----------|-----|
| **0-2** | País | Continental |
| **3-5** | Región/Ciudad | Metropolitano |
| **6-8** | Barrio | Análisis urbano |
| **9-11** | Cuadra | Micro-localización |
| **12-15** | Edificio | Indoor |

**💡 Recomendación para Mendoza**: Resolución 8 (barrios)

### Uso de H3

```python
import h3

# Convertir lat/lon a H3
lat, lon = -32.889458, -68.845839
h3_index = h3.latlng_to_cell(lat, lon, resolution=8)
print(f"H3 index: {h3_index}")

# Obtener coordenadas del centro del hexágono
center_lat, center_lon = h3.cell_to_latlng(h3_index)

# Obtener hexágonos vecinos
neighbors = h3.grid_disk(h3_index, k=1)  # k=1: vecinos inmediatos

# Aplicar a DataFrame
df['h3'] = df.apply(lambda row: h3.latlng_to_cell(
    row['lat'], row['lon'], resolution=8
), axis=1)
```

---

## 4️⃣ Choropleth con H3

```python
# Agregar ventas por zona H3
zonas = df.groupby('h3').agg({
    'ventas': 'sum',
    'clientes': 'nunique'
}).reset_index()

# Obtener coordenadas de hexágonos
zonas['lat'] = zonas['h3'].apply(lambda x: h3.cell_to_latlng(x)[0])
zonas['lon'] = zonas['h3'].apply(lambda x: h3.cell_to_latlng(x)[1])

# Visualizar con scatter (color por ventas)
fig = px.scatter_mapbox(
    zonas,
    lat='lat',
    lon='lon',
    color='ventas',
    size='ventas',
    hover_data=['clientes'],
    color_continuous_scale='YlOrRd',
    title='Zonas Calientes de Ventas (H3)',
    mapbox_style='carto-positron',
    zoom=11
)
fig.show()
```

---

## 5️⃣ Casos de Uso

### **1. Optimización de Ubicación**

```python
# Identificar zonas con alta demanda y baja cobertura
cobertura = df.groupby('h3').agg({
    'sucursal_mas_cercana_km': 'min',
    'demanda_estimada': 'sum'
}).reset_index()

# Zonas sin cobertura adecuada
oportunidades = cobertura[
    (cobertura['sucursal_mas_cercana_km'] > 2) & 
    (cobertura['demanda_estimada'] > 10000)
]
```

### **2. Análisis de Rutas**

```python
# Agrupar órdenes por zona para delivery
rutas = df.groupby(['h3', 'zona']).agg({
    'orden_id': 'count',
    'lat': 'mean',
    'lon': 'mean'
}).reset_index()

# Optimizar secuencia de paradas
```

### **3. Segmentación Geográfica**

```python
# Caracterizar zonas
features_zona = df.groupby('h3').agg({
    'ingreso_promedio': 'mean',
    'edad_promedio': 'mean',
    'ticket_promedio': 'mean',
    'frecuencia_compra': 'mean'
})

# Clustering espacial
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=4)
features_zona['segmento'] = kmeans.fit_predict(features_zona)
```

---

## ✅ Resumen

### Mejores Prácticas

✅ **Resolución H3**: Elegir según escala del análisis  
✅ **Performance**: Agregar datos antes de mapear  
✅ **Colores**: Escalas apropiadas (secuenciales, divergentes)  
✅ **Zoom**: Ajustar para contexto adecuado  
✅ **Tooltips**: Información relevante de negocio

---

### 🚀 Próximos Pasos

* Practica con ejercicios del Módulo 02
* Aplica en proyecto final geoespacial
* Integra con dashboards del Módulo 04

---

**Universidad del Aconcagua - Mendoza 🇦🇷**